In [1]:
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
from folium.plugins import TimestampedGeoJson

history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
stations_toulouse = pd.read_parquet("stations_toulouse.parquet")
history_toulouse_validation = pd.read_parquet("history_toulouse_validation.parquet")
history_toulouse_apprentissage = pd.read_parquet("history_toulouse_apprentissage.parquet")

In [2]:
station1 = "00001 - POIDS DE L'HUILE"
donnees1 = history_toulouse_apprentissage.loc[station1]
dates = donnees1.index.get_level_values("commit_at")
donnees1["heure_minute"] = dates.strftime("%H:%M")

moyqh = (
    donnees1
    .groupby(["heure_minute"])[["bikes"]]
        .mean()

)
moyqh

,bikes
heure_minute,
00:00,2.396884
00:15,2.318057
00:30,2.211732
00:45,2.153987
01:00,2.055912
...,...
22:45,3.341888
23:00,3.129239
23:15,2.868928


In [3]:
station = history_toulouse_apprentissage.index.get_level_values("station").unique()
donnees = history_toulouse_apprentissage.reset_index()
donnees["heure_minute"] = donnees["commit_at"].dt.strftime("%H:%M")

moyqhs = donnees.groupby(["station", "heure_minute"])["bikes"].mean().round()

In [4]:
moyqhs

station                   heure_minute
00001 - POIDS DE L'HUILE  00:00           2.0
                          00:15           2.0
                          00:30           2.0
                          00:45           2.0
                          01:00           2.0
                                         ... 
TEST BEE                  22:45           1.0
                          23:00           1.0
                          23:15           1.0
                          23:30           1.0
                          23:45           1.0
Name: bikes, Length: 76506, dtype: float64

In [5]:
history_toulouse_moyenne_heure_minute = history_toulouse_test.copy()


In [6]:
heure_station = history_toulouse_moyenne_heure_minute.index.get_level_values("commit_at")
heure_station

DatetimeIndex(['2026-07-31 21:45:00', '2026-07-31 23:45:00',
               '2025-11-10 11:45:00', '2026-07-31 21:45:00',
               '2025-01-05 05:45:00', '2025-11-06 07:15:00',
               '2024-10-31 11:00:00', '2026-07-31 22:30:00',
               '2025-01-05 05:45:00', '2025-01-05 05:45:00',
               ...
               '2026-07-21 10:00:00', '2026-07-21 10:00:00',
               '2025-08-24 18:30:00', '2026-03-23 21:15:00',
               '2026-03-23 20:15:00', '2025-08-22 19:15:00',
               '2025-08-05 10:15:00', '2025-08-05 11:00:00',
               '2025-08-01 12:15:00', '2026-02-12 16:00:00'],
              dtype='datetime64[us]', name='commit_at', length=800, freq=None)

In [7]:

horizons_cols = [
    'bikes_+15min', 'bikes_+30min', 'bikes_+45min', 'bikes_+60min', 
    'bikes_+75min', 'bikes_+90min', 'bikes_+105min', 'bikes_+120min'
]

for idx, row in history_toulouse_moyenne_heure_minute.iterrows():
    station, commit_at = idx
    for j, col_name in enumerate(horizons_cols):
        future_time = commit_at + pd.Timedelta(minutes=15 * (j + 1))
        heure_str = future_time.strftime("%H:%M")
        try:
            valeur_moyenne = moyqhs.loc[(station, heure_str)]
            history_toulouse_moyenne_heure_minute.loc[idx, col_name] = valeur_moyenne
        except KeyError:
            pass

history_toulouse_moyenne_heure_minute = history_toulouse_moyenne_heure_minute.fillna(0)


In [8]:
history_toulouse_moyenne_heure_minute

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,4.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0
00002 - LAFAYETTE,2026-07-31 23:45:00,5.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11.0,11.0,11.0,10.0,10.0,10.0,9.0,9.0
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,4.0,4.0,4.0,3.0,3.0,2.0,2.0,2.0
00003 - POMME,2025-01-05 05:45:00,8.0,8.0,8.0,8.0,8.0,9.0,9.0,9.0
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
print("Nombre de valeurs manquantes :", history_toulouse_moyenne_heure_minute.isna().sum())
print("Nombre de valeurs manquantes :", history_toulouse_validation.isna().sum())
history_toulouse_validation = history_toulouse_validation.fillna(0)

Nombre de valeurs manquantes : horizon
bikes_+15min     0
bikes_+30min     0
bikes_+45min     0
bikes_+60min     0
bikes_+75min     0
bikes_+90min     0
bikes_+105min    0
bikes_+120min    0
dtype: int64
Nombre de valeurs manquantes : horizon
bikes_+15min     0
bikes_+30min     0
bikes_+45min     1
bikes_+60min     1
bikes_+75min     2
bikes_+90min     2
bikes_+105min    2
bikes_+120min    3
dtype: int64


In [13]:
rmse = np.sqrt((
    (history_toulouse_moyenne_heure_minute.values - history_toulouse_validation.values)**2
).mean())
print(f'RMSE',rmse)

RMSE 5.63684863199288
